In [1]:
!pip install -q nltk pandas numpy matplotlib indic-transliteration

In [2]:
import re
import unicodedata
import string
from collections import Counter

import numpy as np
import pandas as pd
import nltk

# NLTK resources
nltk.download('indian')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('udhr')

from nltk.corpus import indian
from nltk.tokenize import wordpunct_tokenize

[nltk_data] Downloading package indian to
[nltk_data]     C:\Users\pragy\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\indian.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\pragy\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\pragy\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package udhr to
[nltk_data]     C:\Users\pragy\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\udhr.zip.


In [3]:
# Display available files in the Indian corpus
print("Available Indian corpus files:")
print(indian.fileids())

Available Indian corpus files:
['bangla.pos', 'hindi.pos', 'marathi.pos', 'telugu.pos']


In [4]:
# Load Hindi sentences from NLTK Indian corpus
hindi_sents = indian.sents('hindi.pos')

print("Total Hindi sentences available:", len(hindi_sents))

# Extract at least 500 sentences
sentences = hindi_sents[:500]

print("\nNumber of selected sentences:", len(sentences))

# Display first 5 sentences
for i, sent in enumerate(sentences[:5], 1):
    print(f"{i}. {' '.join(sent)}")

Total Hindi sentences available: 540

Number of selected sentences: 500
1. पूर्ण प्रतिबंध हटाओ : इराक
2. संयुक्त राष्ट्र ।
3. इराक के विदेश मंत्री ने अमरीका के उस प्रस्ताव का मजाक उड़ाया है , जिसमें अमरीका ने संयुक्त राष्ट्र के प्रतिबंधों को इराकी नागरिकों के लिए कम हानिकारक बनाने के लिए कहा है ।
4. विदेश मंत्री का कहना है कि चूंकि बगदाद संयुक्त राष्ट्र की मांगों का पालन करते हुए अपने भारी विनाशकारी हथियारों को नष्ट कर रहा है ।
5. लिहाजा प्रतिबंधों को पूर्ण रूप से उठा दिया जाना चाहिए ।


In [5]:
# Convert each sentence back to text
raw_sentences = [' '.join(sentence) for sentence in sentences]

# Tokenize each sentence
tokenized_sentences = [
    wordpunct_tokenize(sentence)
    for sentence in raw_sentences
]

print("Original sentence:")
print(raw_sentences[0])

print("\nTokenized sentence:")
print(tokenized_sentences[0])

Original sentence:
पूर्ण प्रतिबंध हटाओ : इराक

Tokenized sentence:
['प', 'ू', 'र', '्', 'ण', 'प', '्', 'रत', 'ि', 'ब', 'ं', 'ध', 'हट', 'ा', 'ओ', ':', 'इर', 'ा', 'क']


In [6]:
def preprocess_indic_tokens(tokens):
    cleaned = []

    for token in tokens:
        # Unicode normalization
        token = unicodedata.normalize("NFC", token)

        # Remove punctuation
        if token in string.punctuation:
            continue

        # Remove tokens containing only punctuation/symbols
        if not re.search(r'\w', token, flags=re.UNICODE):
            continue

        token = token.strip()

        if token:
            cleaned.append(token)

    return cleaned


processed_sentences = [
    preprocess_indic_tokens(tokens)
    for tokens in tokenized_sentences
]

print("Before preprocessing:")
print(tokenized_sentences[0])

print("\nAfter preprocessing:")
print(processed_sentences[0])

Before preprocessing:
['प', 'ू', 'र', '्', 'ण', 'प', '्', 'रत', 'ि', 'ब', 'ं', 'ध', 'हट', 'ा', 'ओ', ':', 'इर', 'ा', 'क']

After preprocessing:
['प', 'र', 'ण', 'प', 'रत', 'ब', 'ध', 'हट', 'ओ', 'इर', 'क']


In [7]:
# Flatten all sentences
all_tokens = [
    token
    for sentence in processed_sentences
    for token in sentence
]

# Total number of tokens
total_tokens = len(all_tokens)

# Vocabulary
vocabulary = set(all_tokens)

# Vocabulary size
vocabulary_size = len(vocabulary)

print("========== CORPUS STATISTICS ==========")
print("Number of sentences :", len(processed_sentences))
print("Total tokens        :", total_tokens)
print("Vocabulary size     :", vocabulary_size)

========== CORPUS STATISTICS ==========
Number of sentences : 500
Total tokens        : 15183
Vocabulary size     : 775


In [8]:
# Count token frequencies
token_frequency = Counter(all_tokens)

# Top 20 frequent tokens
top_tokens = token_frequency.most_common(20)

print("========== TOP 20 FREQUENT TOKENS ==========")

for token, frequency in top_tokens:
    print(f"{token:<20} {frequency}")

========== TOP 20 FREQUENT TOKENS ==========
क                    1616
र                    1004
न                    794
म                    714
स                    691
ह                    629
ल                    579
त                    536
य                    527
द                    495
व                    418
प                    415
ज                    300
ट                    290
ग                    283
ब                    225
च                    191
श                    185
पर                   144
थ                    139


In [9]:
statistics = pd.DataFrame({
    "Metric": [
        "Number of Sentences",
        "Total Tokens",
        "Vocabulary Size",
        "Average Tokens per Sentence"
    ],
    "Value": [
        len(processed_sentences),
        total_tokens,
        vocabulary_size,
        round(total_tokens / len(processed_sentences), 2)
    ]
})

statistics

,Metric,Value
0,Number of Sentences,500.00
1,Total Tokens,15183.00
2,Vocabulary Size,775.00
3,Average Tokens per Sentence,30.37


In [10]:
top_tokens_df = pd.DataFrame(
    top_tokens,
    columns=["Token", "Frequency"]
)

top_tokens_df

,Token,Frequency
0,क,1616
1,र,1004
2,न,794
3,म,714
4,स,691
5,ह,629
6,ल,579
7,त,536
8,य,527
9,द,495


### Challenges of Tokenization and Preprocessing for Indic Languages

1. Indic languages contain complex Unicode characters and combining marks.
2. A single visible character may consist of multiple Unicode code points.
3. Word boundaries are not always as straightforward as in English.
4. Different spellings and Unicode representations may represent the same text.
5. Punctuation and symbols can be attached directly to words.
6. Code-mixed text may contain both Indic and Roman-script words.
7. Transliteration creates additional spelling variations.
8. Morphological richness can result in many different word forms.
9. Normalization is therefore important before NLP tasks such as classification,
   sentiment analysis and machine translation.

In [12]:
noisy_texts = [
    "नमस्ते   दुनिया!!!",
    "आज   मौसम बहुत अच्छा है....",
    "मुझे    Python  सीखना है!!!",
    "यह    एक बहुत अच्छा   दिन है।",
    "आप कैसे हैं????",
    "मैं   कॉलेज जा रही हूँ!!!",
    "मुझे AI बहुत पसंद है!!!",
    "आज का तापमान  30°C है।",
    "बहुत बहुत बहुत अच्छा!!!!",
    "कृपया   जल्दी   आओ...",
    "HELLO   दुनिया!!!",
    "Python   बहुत   useful है!!!",
    "मेरा   नाम   Pragya है।",
    "यह text बहुत अच्छा है!!!!",
    "क्या तुमने assignment किया????",
    "आज   class   बहुत interesting थी!!!",
    "मुझे machine learning सीखनी है...",
    "वाह!!! यह बहुत सुंदर है।",
    "GOOD   MORNING   सभी को!!!",
    "NLP सीखना बहुत मजेदार है!!!!"
]

print("Number of noisy samples:", len(noisy_texts))

for text in noisy_texts[:5]:
    print(text)

Number of noisy samples: 20
नमस्ते   दुनिया!!!
आज   मौसम बहुत अच्छा है....
मुझे    Python  सीखना है!!!
यह    एक बहुत अच्छा   दिन है।
आप कैसे हैं????


In [15]:
def normalize_text(text):
    # 1. Unicode normalization
    text = unicodedata.normalize("NFC", text)

    # 2. Normalize different dash characters
    text = re.sub(r'[‐-‒–—−]', '-', text)

    # 3. Normalize quotation marks
    text = re.sub(r'[“”„‟]', '"', text)
    text = re.sub(r"[‘’‚‛]", "'", text)

    # 4. Reduce repeated punctuation
    text = re.sub(r'([!?।,.])\1+', r'\1', text)

    # 5. Remove spaces before punctuation
    text = re.sub(r'\s+([!?।,.])', r'\1', text)

    # 6. Normalize repeated whitespace
    text = re.sub(r'\s+', ' ', text)

    # 7. Lowercase Roman-script words
    text = re.sub(
        r'[A-Za-z]+',
        lambda m: m.group(0).lower(),
        text
    )

    # 8. Strip leading/trailing whitespace
    text = text.strip()

    return text

In [16]:
normalized_texts = [
    normalize_text(text)
    for text in noisy_texts
]

normalization_df = pd.DataFrame({
    "Original Text": noisy_texts,
    "Normalized Text": normalized_texts
})

pd.set_option('display.max_colwidth', 100)

normalization_df

,Original Text,Normalized Text
0,नमस्ते दुनिया!!!,नमस्ते दुनिया!
1,आज मौसम बहुत अच्छा है....,आज मौसम बहुत अच्छा है.
2,मुझे Python सीखना है!!!,मुझे python सीखना है!
3,यह एक बहुत अच्छा दिन है।,यह एक बहुत अच्छा दिन है।
4,आप कैसे हैं????,आप कैसे हैं?
5,मैं कॉलेज जा रही हूँ!!!,मैं कॉलेज जा रही हूँ!
6,मुझे AI बहुत पसंद है!!!,मुझे ai बहुत पसंद है!
7,आज का तापमान 30°C है।,आज का तापमान 30°c है।
8,बहुत बहुत बहुत अच्छा!!!!,बहुत बहुत बहुत अच्छा!
9,कृपया जल्दी आओ...,कृपया जल्दी आओ.


In [17]:
# Example of Unicode normalization

text = "नमस्ते"

print("Original:", text)
print("NFC     :", unicodedata.normalize("NFC", text))
print("NFD     :", unicodedata.normalize("NFD", text))

print("\nUnicode representation:")
for char in text:
    print(char, hex(ord(char)))

Original: नमस्ते
NFC     : नमस्ते
NFD     : नमस्ते

Unicode representation:
न 0x928
म 0x92e
स 0x938
् 0x94d
त 0x924
े 0x947


### Importance of Text Normalization

Text normalization reduces unnecessary variations in text.

For example:

"HELLO", "Hello", and "hello"

can be treated as the same word after case normalization.

Similarly:

"नमस्ते!!!"
"नमस्ते!!"
"नमस्ते!"

can be normalized to:

"नमस्ते!"

Normalization helps NLP systems by:

- reducing vocabulary size
- improving consistency
- reducing noise
- improving model performance
- making text comparison easier
- improving downstream tasks such as classification,
  information retrieval and machine translation

In [18]:
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

In [19]:
script_samples = [
    ("Hindi", "नमस्ते दुनिया"),
    ("Hindi", "मेरा नाम प्रज्ञा है"),
    ("Hindi", "आज मौसम अच्छा है"),
    ("Hindi", "मुझे मशीन लर्निंग पसंद है"),
    ("Hindi", "भारत एक सुंदर देश है"),
    ("Hindi", "मैं कॉलेज जा रही हूँ"),
    ("Hindi", "यह मेरा प्रोजेक्ट है"),
    ("Hindi", "कृत्रिम बुद्धिमत्ता भविष्य है"),
    ("Hindi", "मुझे Python सीखना पसंद है"),
    ("Hindi", "आज मेरी क्लास है"),

    ("Bengali", "নমস্কার বিশ্ব"),
    ("Bengali", "আমার নাম প্রজ্ঞা"),
    ("Bengali", "আজ আবহাওয়া ভালো"),
    ("Bengali", "আমি কলেজে যাচ্ছি"),
    ("Bengali", "মেশিন লার্নিং খুব আকর্ষণীয়"),
    ("Bengali", "ভারত একটি সুন্দর দেশ"),
    ("Bengali", "এটি আমার প্রজেক্ট"),
    ("Bengali", "কৃত্রিম বুদ্ধিমত্তা ভবিষ্যৎ"),
    ("Bengali", "আমি পাইথন শিখছি"),
    ("Bengali", "আজ আমার ক্লাস আছে")
]

script_df = pd.DataFrame(
    script_samples,
    columns=["Language", "Original"]
)

script_df

,Language,Original
0,Hindi,नमस्ते दुनिया
1,Hindi,मेरा नाम प्रज्ञा है
2,Hindi,आज मौसम अच्छा है
3,Hindi,मुझे मशीन लर्निंग पसंद है
4,Hindi,भारत एक सुंदर देश है
5,Hindi,मैं कॉलेज जा रही हूँ
6,Hindi,यह मेरा प्रोजेक्ट है
7,Hindi,कृत्रिम बुद्धिमत्ता भविष्य है
8,Hindi,मुझे Python सीखना पसंद है
9,Hindi,आज मेरी क्लास है


In [20]:
def detect_script(text):
    devanagari = len(re.findall(r'[\u0900-\u097F]', text))
    bengali = len(re.findall(r'[\u0980-\u09FF]', text))

    if devanagari > 0:
        return "Devanagari"
    elif bengali > 0:
        return "Bengali"
    else:
        return "Other/Latin"


script_df["Detected Script"] = script_df["Original"].apply(detect_script)

script_df

,Language,Original,Detected Script
0,Hindi,नमस्ते दुनिया,Devanagari
1,Hindi,मेरा नाम प्रज्ञा है,Devanagari
2,Hindi,आज मौसम अच्छा है,Devanagari
3,Hindi,मुझे मशीन लर्निंग पसंद है,Devanagari
4,Hindi,भारत एक सुंदर देश है,Devanagari
5,Hindi,मैं कॉलेज जा रही हूँ,Devanagari
6,Hindi,यह मेरा प्रोजेक्ट है,Devanagari
7,Hindi,कृत्रिम बुद्धिमत्ता भविष्य है,Devanagari
8,Hindi,मुझे Python सीखना पसंद है,Devanagari
9,Hindi,आज मेरी क्लास है,Devanagari


In [21]:
def transliterate_text(language, text):
    if language == "Hindi":
        return transliterate(
            text,
            sanscript.DEVANAGARI,
            sanscript.ITRANS
        )

    elif language == "Bengali":
        return transliterate(
            text,
            sanscript.BENGALI,
            sanscript.ITRANS
        )

    return text


script_df["Transliterated"] = script_df.apply(
    lambda row: transliterate_text(
        row["Language"],
        row["Original"]
    ),
    axis=1
)

script_df

,Language,Original,Detected Script,Transliterated
0,Hindi,नमस्ते दुनिया,Devanagari,namaste duniyA
1,Hindi,मेरा नाम प्रज्ञा है,Devanagari,merA nAma praj~nA hai
2,Hindi,आज मौसम अच्छा है,Devanagari,Aja mausama achChA hai
3,Hindi,मुझे मशीन लर्निंग पसंद है,Devanagari,mujhe mashIna larniMga pasaMda hai
4,Hindi,भारत एक सुंदर देश है,Devanagari,bhArata eka suMdara desha hai
5,Hindi,मैं कॉलेज जा रही हूँ,Devanagari,maiM kaॉleja jA rahI hU.N
6,Hindi,यह मेरा प्रोजेक्ट है,Devanagari,yaha merA projekTa hai
7,Hindi,कृत्रिम बुद्धिमत्ता भविष्य है,Devanagari,kRRitrima buddhimattA bhaviShya hai
8,Hindi,मुझे Python सीखना पसंद है,Devanagari,mujhe Python sIkhanA pasaMda hai
9,Hindi,आज मेरी क्लास है,Devanagari,Aja merI klAsa hai


In [22]:
script_df["Normalized"] = script_df["Original"].apply(normalize_text)

script_df = script_df[
    ["Language", "Detected Script", "Original",
     "Normalized", "Transliterated"]
]

script_df

,Language,Detected Script,Original,Normalized,Transliterated
0,Hindi,Devanagari,नमस्ते दुनिया,नमस्ते दुनिया,namaste duniyA
1,Hindi,Devanagari,मेरा नाम प्रज्ञा है,मेरा नाम प्रज्ञा है,merA nAma praj~nA hai
2,Hindi,Devanagari,आज मौसम अच्छा है,आज मौसम अच्छा है,Aja mausama achChA hai
3,Hindi,Devanagari,मुझे मशीन लर्निंग पसंद है,मुझे मशीन लर्निंग पसंद है,mujhe mashIna larniMga pasaMda hai
4,Hindi,Devanagari,भारत एक सुंदर देश है,भारत एक सुंदर देश है,bhArata eka suMdara desha hai
5,Hindi,Devanagari,मैं कॉलेज जा रही हूँ,मैं कॉलेज जा रही हूँ,maiM kaॉleja jA rahI hU.N
6,Hindi,Devanagari,यह मेरा प्रोजेक्ट है,यह मेरा प्रोजेक्ट है,yaha merA projekTa hai
7,Hindi,Devanagari,कृत्रिम बुद्धिमत्ता भविष्य है,कृत्रिम बुद्धिमत्ता भविष्य है,kRRitrima buddhimattA bhaviShya hai
8,Hindi,Devanagari,मुझे Python सीखना पसंद है,मुझे python सीखना पसंद है,mujhe Python sIkhanA pasaMda hai
9,Hindi,Devanagari,आज मेरी क्लास है,आज मेरी क्लास है,Aja merI klAsa hai


In [23]:
script_statistics = script_df.groupby(
    ["Language", "Detected Script"]
).size().reset_index(name="Number of Samples")

script_statistics

,Language,Detected Script,Number of Samples
0,Bengali,Bengali,10
1,Hindi,Devanagari,10


### Role of Script Normalization in Multilingual NLP

Script normalization converts different writing representations into a
consistent form.

For example:

Hindi:
नमस्ते

can be transliterated into Roman representation:

namaste

Benefits:

1. Makes multilingual text easier to compare.
2. Helps handle different scripts.
3. Supports cross-lingual NLP.
4. Helps in information retrieval.
5. Helps machine translation systems.
6. Useful for code-mixed and transliterated social-media text.
7. Reduces variation caused by multiple writing systems.

In [25]:
parallel_data = [
    ("मैं कॉलेज जा रही हूँ।", "I am going to college."),
    ("मुझे मशीन लर्निंग पसंद है।", "I like machine learning."),
    ("आज मौसम बहुत अच्छा है।", "The weather is very nice today."),
    ("मेरा नाम प्रज्ञा है।", "My name is Pragya."),
    ("भारत एक सुंदर देश है।", "India is a beautiful country."),
    ("मैं रोज पढ़ाई करती हूँ।", "I study every day."),
    ("यह मेरा प्रोजेक्ट है।", "This is my project."),
    ("मुझे Python सीखना है।", "I want to learn Python."),
    ("आज मेरी क्लास है।", "I have a class today."),
    ("वह मेरा दोस्त है।", "He is my friend."),
    ("मुझे किताब पढ़ना पसंद है।", "I like reading books."),
    ("हम साथ में काम करते हैं।", "We work together."),
    ("कृपया दरवाजा बंद करें।", "Please close the door."),
    ("मुझे पानी चाहिए।", "I need water."),
    ("आज परीक्षा है।", "There is an exam today."),
    ("मैं अपना assignment पूरा कर रही हूँ।", "I am completing my assignment."),
    ("यह बहुत महत्वपूर्ण है।", "This is very important."),
    ("मुझे artificial intelligence सीखनी है।", "I want to learn artificial intelligence."),
    ("वह बाजार जा रहा है।", "He is going to the market."),
    ("हम कल मिलेंगे।", "We will meet tomorrow.")
]

parallel_df = pd.DataFrame(
    parallel_data,
    columns=["Hindi", "English"]
)

print("Number of parallel sentence pairs:", len(parallel_df))

parallel_df

Number of parallel sentence pairs: 20


,Hindi,English
0,मैं कॉलेज जा रही हूँ।,I am going to college.
1,मुझे मशीन लर्निंग पसंद है।,I like machine learning.
2,आज मौसम बहुत अच्छा है।,The weather is very nice today.
3,मेरा नाम प्रज्ञा है।,My name is Pragya.
4,भारत एक सुंदर देश है।,India is a beautiful country.
5,मैं रोज पढ़ाई करती हूँ।,I study every day.
6,यह मेरा प्रोजेक्ट है।,This is my project.
7,मुझे Python सीखना है।,I want to learn Python.
8,आज मेरी क्लास है।,I have a class today.
9,वह मेरा दोस्त है।,He is my friend.


In [26]:
parallel_df["Hindi Tokens"] = parallel_df["Hindi"].apply(
    wordpunct_tokenize
)

parallel_df["English Tokens"] = parallel_df["English"].apply(
    wordpunct_tokenize
)

parallel_df["Hindi Length"] = parallel_df["Hindi Tokens"].apply(len)
parallel_df["English Length"] = parallel_df["English Tokens"].apply(len)

parallel_df[
    ["Hindi", "English", "Hindi Tokens",
     "English Tokens", "Hindi Length", "English Length"]
]

,Hindi,English,Hindi Tokens,English Tokens,Hindi Length,English Length
0,मैं कॉलेज जा रही हूँ।,I am going to college.,"[म, ैं, क, ॉ, ल, े, ज, ज, ा, रह, ी, ह, ूँ।]","[I, am, going, to, college, .]",13,6
1,मुझे मशीन लर्निंग पसंद है।,I like machine learning.,"[म, ु, झ, े, मश, ी, न, लर, ्, न, िं, ग, पस, ं, द, ह, ै।]","[I, like, machine, learning, .]",17,5
2,आज मौसम बहुत अच्छा है।,The weather is very nice today.,"[आज, म, ौ, सम, बह, ु, त, अच, ्, छ, ा, ह, ै।]","[The, weather, is, very, nice, today, .]",13,7
3,मेरा नाम प्रज्ञा है।,My name is Pragya.,"[म, े, र, ा, न, ा, म, प, ्, रज, ्, ञ, ा, ह, ै।]","[My, name, is, Pragya, .]",15,5
4,भारत एक सुंदर देश है।,India is a beautiful country.,"[भ, ा, रत, एक, स, ुं, दर, द, े, श, ह, ै।]","[India, is, a, beautiful, country, .]",12,6
5,मैं रोज पढ़ाई करती हूँ।,I study every day.,"[म, ैं, र, ो, ज, पढ, ़ा, ई, करत, ी, ह, ूँ।]","[I, study, every, day, .]",12,5
6,यह मेरा प्रोजेक्ट है।,This is my project.,"[यह, म, े, र, ा, प, ्, र, ो, ज, े, क, ्, ट, ह, ै।]","[This, is, my, project, .]",16,5
7,मुझे Python सीखना है।,I want to learn Python.,"[म, ु, झ, े, Python, स, ी, खन, ा, ह, ै।]","[I, want, to, learn, Python, .]",11,6
8,आज मेरी क्लास है।,I have a class today.,"[आज, म, े, र, ी, क, ्, ल, ा, स, ह, ै।]","[I, have, a, class, today, .]",12,6
9,वह मेरा दोस्त है।,He is my friend.,"[वह, म, े, र, ा, द, ो, स, ्, त, ह, ै।]","[He, is, my, friend, .]",12,5


In [27]:
print("========== PARALLEL SENTENCE ALIGNMENT ==========\n")

for i, row in parallel_df.iterrows():
    print(f"Pair {i+1}")
    print("Hindi   :", row["Hindi"])
    print("English :", row["English"])
    print("-" * 60)

========== PARALLEL SENTENCE ALIGNMENT ==========

Pair 1
Hindi   : मैं कॉलेज जा रही हूँ।
English : I am going to college.
------------------------------------------------------------
Pair 2
Hindi   : मुझे मशीन लर्निंग पसंद है।
English : I like machine learning.
------------------------------------------------------------
Pair 3
Hindi   : आज मौसम बहुत अच्छा है।
English : The weather is very nice today.
------------------------------------------------------------
Pair 4
Hindi   : मेरा नाम प्रज्ञा है।
English : My name is Pragya.
------------------------------------------------------------
Pair 5
Hindi   : भारत एक सुंदर देश है।
English : India is a beautiful country.
------------------------------------------------------------
Pair 6
Hindi   : मैं रोज पढ़ाई करती हूँ।
English : I study every day.
------------------------------------------------------------
Pair 7
Hindi   : यह मेरा प्रोजेक्ट है।
English : This is my project.
------------------------------------------------------------
Pair

In [28]:
# Flatten Hindi tokens
hindi_tokens = [
    token
    for tokens in parallel_df["Hindi Tokens"]
    for token in tokens
    if token not in string.punctuation
]

# Flatten English tokens
english_tokens = [
    token.lower()
    for tokens in parallel_df["English Tokens"]
    for token in tokens
    if token not in string.punctuation
]

hindi_vocab = set(hindi_tokens)
english_vocab = set(english_tokens)

print("========== PARALLEL CORPUS STATISTICS ==========")

print("Hindi total tokens    :", len(hindi_tokens))
print("Hindi vocabulary size :", len(hindi_vocab))

print("\nEnglish total tokens    :", len(english_tokens))
print("English vocabulary size :", len(english_vocab))

========== PARALLEL CORPUS STATISTICS ==========
Hindi total tokens    : 259
Hindi vocabulary size : 79

English total tokens    : 90
English vocabulary size : 55


In [29]:
length_statistics = pd.DataFrame({
    "Language": ["Hindi", "English"],
    "Average Sentence Length": [
        parallel_df["Hindi Length"].mean(),
        parallel_df["English Length"].mean()
    ],
    "Maximum Sentence Length": [
        parallel_df["Hindi Length"].max(),
        parallel_df["English Length"].max()
    ],
    "Minimum Sentence Length": [
        parallel_df["Hindi Length"].min(),
        parallel_df["English Length"].min()
    ]
})

length_statistics

,Language,Average Sentence Length,Maximum Sentence Length,Minimum Sentence Length
0,Hindi,12.95,18,8
1,English,5.50,7,4


In [30]:
code_mixed = [
    "Mujhe Python सीखना है.",
    "Aaj class बहुत interesting थी.",
    "Mera project machine learning पर है.",
    "Please मुझे assignment भेज दो.",
    "Kal मेरा interview है.",
    "I am college जा रही हूँ.",
    "Mujhe AI में interest है.",
    "Tumne project complete किया?",
    "Today मेरी class online है.",
    "Please जल्दी reply करो."
]

code_mixed_df = pd.DataFrame(
    {"Original Code-Mixed Text": code_mixed}
)

code_mixed_df

,Original Code-Mixed Text
0,Mujhe Python सीखना है.
1,Aaj class बहुत interesting थी.
2,Mera project machine learning पर है.
3,Please मुझे assignment भेज दो.
4,Kal मेरा interview है.
5,I am college जा रही हूँ.
6,Mujhe AI में interest है.
7,Tumne project complete किया?
8,Today मेरी class online है.
9,Please जल्दी reply करो.


In [31]:
def detect_token_language(token):
    # Devanagari Unicode range
    if re.search(r'[\u0900-\u097F]', token):
        return "Hindi"

    # English/Roman alphabet
    elif re.fullmatch(r'[A-Za-z]+', token):
        return "English"

    else:
        return "Other"


def analyze_code_mixed(text):
    tokens = wordpunct_tokenize(text)

    result = []

    for token in tokens:
        if token.strip():
            result.append(
                (token, detect_token_language(token))
            )

    return result


code_mixed_df["Token Analysis"] = (
    code_mixed_df["Original Code-Mixed Text"]
    .apply(analyze_code_mixed)
)

code_mixed_df

,Original Code-Mixed Text,Token Analysis
0,Mujhe Python सीखना है.,"[(Mujhe, English), (Python, English), (स, Hindi), (ी, Hindi), (खन, Hindi), (ा, Hindi), (ह, Hindi..."
1,Aaj class बहुत interesting थी.,"[(Aaj, English), (class, English), (बह, Hindi), (ु, Hindi), (त, Hindi), (interesting, English), ..."
2,Mera project machine learning पर है.,"[(Mera, English), (project, English), (machine, English), (learning, English), (पर, Hindi), (ह, ..."
3,Please मुझे assignment भेज दो.,"[(Please, English), (म, Hindi), (ु, Hindi), (झ, Hindi), (े, Hindi), (assignment, English), (भ, H..."
4,Kal मेरा interview है.,"[(Kal, English), (म, Hindi), (े, Hindi), (र, Hindi), (ा, Hindi), (interview, English), (ह, Hindi..."
5,I am college जा रही हूँ.,"[(I, English), (am, English), (college, English), (ज, Hindi), (ा, Hindi), (रह, Hindi), (ी, Hindi..."
6,Mujhe AI में interest है.,"[(Mujhe, English), (AI, English), (म, Hindi), (ें, Hindi), (interest, English), (ह, Hindi), (ै.,..."
7,Tumne project complete किया?,"[(Tumne, English), (project, English), (complete, English), (क, Hindi), (ि, Hindi), (य, Hindi), ..."
8,Today मेरी class online है.,"[(Today, English), (म, Hindi), (े, Hindi), (र, Hindi), (ी, Hindi), (class, English), (online, En..."
9,Please जल्दी reply करो.,"[(Please, English), (जल, Hindi), (्, Hindi), (द, Hindi), (ी, Hindi), (reply, English), (कर, Hind..."


In [32]:
for i, text in enumerate(code_mixed, 1):

    print(f"\nSentence {i}: {text}")

    tokens = wordpunct_tokenize(text)

    for token in tokens:
        if token.strip():
            language = detect_token_language(token)
            print(f"{token:<15} -> {language}")


Sentence 1: Mujhe Python सीखना है.
Mujhe           -> English
Python          -> English
स               -> Hindi
ी               -> Hindi
खन              -> Hindi
ा               -> Hindi
ह               -> Hindi
ै.              -> Hindi

Sentence 2: Aaj class बहुत interesting थी.
Aaj             -> English
class           -> English
बह              -> Hindi
ु               -> Hindi
त               -> Hindi
interesting     -> English
थ               -> Hindi
ी.              -> Hindi

Sentence 3: Mera project machine learning पर है.
Mera            -> English
project         -> English
machine         -> English
learning        -> English
पर              -> Hindi
ह               -> Hindi
ै.              -> Hindi

Sentence 4: Please मुझे assignment भेज दो.
Please          -> English
म               -> Hindi
ु               -> Hindi
झ               -> Hindi
े               -> Hindi
assignment      -> English
भ               -> Hindi
े               -> Hindi
ज               -> Hindi
द   

In [33]:
code_mixed_df["Normalized Text"] = (
    code_mixed_df["Original Code-Mixed Text"]
    .apply(normalize_text)
)

code_mixed_df[
    ["Original Code-Mixed Text", "Normalized Text"]
]

,Original Code-Mixed Text,Normalized Text
0,Mujhe Python सीखना है.,mujhe python सीखना है.
1,Aaj class बहुत interesting थी.,aaj class बहुत interesting थी.
2,Mera project machine learning पर है.,mera project machine learning पर है.
3,Please मुझे assignment भेज दो.,please मुझे assignment भेज दो.
4,Kal मेरा interview है.,kal मेरा interview है.
5,I am college जा रही हूँ.,i am college जा रही हूँ.
6,Mujhe AI में interest है.,mujhe ai में interest है.
7,Tumne project complete किया?,tumne project complete किया?
8,Today मेरी class online है.,today मेरी class online है.
9,Please जल्दी reply करो.,please जल्दी reply करो.


In [34]:
def code_mixed_statistics(text):
    tokens = wordpunct_tokenize(text)

    hindi_count = 0
    english_count = 0
    other_count = 0

    for token in tokens:
        language = detect_token_language(token)

        if language == "Hindi":
            hindi_count += 1
        elif language == "English":
            english_count += 1
        else:
            other_count += 1

    return pd.Series({
        "Hindi Tokens": hindi_count,
        "English Tokens": english_count,
        "Other Tokens": other_count,
        "Total Tokens": len(tokens)
    })


cm_stats = code_mixed_df[
    "Original Code-Mixed Text"
].apply(code_mixed_statistics)

cm_stats

,Hindi Tokens,English Tokens,Other Tokens,Total Tokens
0,6,2,0,8
1,5,3,0,8
2,3,4,0,7
3,9,2,0,11
4,6,2,0,8
5,6,3,0,9
6,4,3,0,7
7,4,3,0,7
8,6,3,0,9
9,6,2,0,8


### Challenges of Parallel and Code-Mixed Text

#### Parallel Corpus Challenges

1. Hindi and English have different word orders.
2. Sentence lengths can differ between languages.
3. One word in one language may require multiple words in another.
4. Accurate sentence alignment is important.
5. Translation may change grammatical structure.
6. Vocabulary size differs across languages.

#### Code-Mixed Text Challenges

1. Multiple languages can occur in the same sentence.
2. Romanized Hindi can be difficult to distinguish from English.
3. Spelling of Romanized Indian languages is inconsistent.
4. Script changes can happen within a single sentence.
5. Standard tokenizers may not correctly handle code-mixed text.
6. Language identification must work at token level.
7. Normalization is difficult because users use different spellings.

Code-mixed NLP therefore requires language identification,
Unicode/script detection, normalization and language-specific processing.

In [35]:
print("=" * 60)
print("       TEXT & SCRIPT NORMALIZATION - SUMMARY")
print("=" * 60)

print("\nQ1 - Indian Language Corpus")
print("Sentences analyzed :", len(processed_sentences))
print("Total tokens       :", total_tokens)
print("Vocabulary size    :", vocabulary_size)

print("\nQ2 - Text Normalization")
print("Noisy samples      :", len(noisy_texts))
print("Normalized samples :", len(normalized_texts))

print("\nQ3 - Script Normalization")
print("Languages           :", script_df["Language"].nunique())
print("Total samples       :", len(script_df))

print("\nQ4 - Parallel Corpus")
print("Parallel pairs      :", len(parallel_df))
print("Code-mixed samples  :", len(code_mixed_df))

print("\n" + "=" * 60)
print("Assignment completed successfully!")
print("=" * 60)

       TEXT & SCRIPT NORMALIZATION - SUMMARY

Q1 - Indian Language Corpus
Sentences analyzed : 500
Total tokens       : 15183
Vocabulary size    : 775

Q2 - Text Normalization
Noisy samples      : 20
Normalized samples : 20

Q3 - Script Normalization
Languages           : 2
Total samples       : 20

Q4 - Parallel Corpus
Parallel pairs      : 20
Code-mixed samples  : 10

Assignment completed successfully!
